# AI-Driven Map Matching and Next-Road-Segment Prediction on Semantically Enriched Road Networks
### A Case Study of Munich

The Pipeline for my project is as follows:

1. Load the TUMDOT-MUC trajectories and keep only cars
2. Turn the local drone coordinates into real world coordinates
3. Get the Munich road network from OpenStreetMap, with semantic road attributes
4. Match each drone position to the road it is really on, to get ground truth
5. Split trajectories into train, validation and test sets (no trajectory leakage)
6. Add artificial GPS noise and downsampling to build the noisy GPS input
7. Run the two traditional baselines: nearest road and FMM (HMM based)
8. Build the candidate table, with the geometric and semantic columns kept separate
9. Train the TensorFlow temporal model (LSTM on geometric features only)
10. Train the TensorFlow semantic model (the same LSTM, plus the semantic columns)
11. Run the feature ablation study
12. Next road segment prediction: Markov baseline and a simple RNN model
13. Compute all evaluation metrics

Both deep learning models in steps 9 and 10 are built by the exact same function, so the
only difference between them is which columns of the candidate table go into the model.

## 0. Setup

In [23]:
!pip install -q osmnx pyproj


In [24]:
import json
import math
import random

import numpy as np
import pandas as pd
import networkx as nx
import osmnx as ox
from pyproj import Transformer
from scipy.spatial import cKDTree
from scipy.stats import wilcoxon
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))


TensorFlow version: 2.20.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 1. Load the TUMDOT-MUC trajectories

The columns in each raw file are, in order:

<b>timestamp, category, track_id, translation_x, translation_y, translation_z,
dimension_x, dimension_y, dimension_z, rotation_x, rotation_y, rotation_z,
velocity_x, velocity_y, velocity_z, acceleration_x, acceleration_y, acceleration_z, interpolated</b>

<b>translation_x</b> and <b>translation_y</b> are the position in metres, offset from a fixed
reference point in UTM zone 32N. <b>rotation_z</b> is the heading in radians.
A new timestamp arrives every 0.08 seconds (12.5 Hz). <b>category</b> is a numeric object
class, TUM's own <b>meta_information.json</b> file (shipped with the full download) has the
exact mapping from number to class name. If that file is not available, category 1 is
used here as the car class.


In [25]:
DATA_DIR = "."                                # folder holding the tumdot_muc_part_*.csv files
META_PATH = "meta_information.json"

# UTM zone 32N reference point the TUMDOT-MUC translations are offset from
# (from the TUM Chair of Traffic Engineering and Control dataset page)
LINK_To_Dataset = "https://www.mos.ed.tum.de/vt/forschung/datensaetze/tumdot-muc/"
UTM_ZONE = 32
UTM_REF_EASTING = 692009.0
UTM_REF_NORTHING = 5338095.0

CAR_CATEGORY = 1   # default, overridden below if meta_information.json is found and says otherwise


In [26]:
import glob

# find every part file and sort them by the number in their name (part_2 after part_1, and
# so on), rather than by plain alphabetical order, which would put part_10 before part_2

part_paths = glob.glob(f"{DATA_DIR}/tumdot_muc_part_*.csv")
part_paths.sort(key=lambda p: int("".join(ch for ch in p.split("part_")[-1] if ch.isdigit())))
print("Found", len(part_paths), "part file(s):")
for p in part_paths:
    print(" ", p)

parts = [pd.read_csv(p) for p in part_paths]
for p, part_df in zip(part_paths, parts):
    print(p, "-> rows:", len(part_df), " timestamp range:", part_df["timestamp"].min(), "to", part_df["timestamp"].max())


Found 1 part file(s):
  ./tumdot_muc_part_1.csv
./tumdot_muc_part_1.csv -> rows: 501728  timestamp range: -313.36 to 286.56


In [27]:
# stitch the parts together into one continuous recording, then drop any row that appears
# in two parts (the same track_id and timestamp recorded at a file boundary), keeping one copy

df = pd.concat(parts, ignore_index=True)
before = len(df)
df = df.drop_duplicates(subset=["track_id", "timestamp"]).reset_index(drop=True)
df = df.sort_values(["track_id", "timestamp"]).reset_index(drop=True)
print("Rows before removing boundary duplicates:", before, " after:", len(df))
print(df.shape)
print(df.head())


Rows before removing boundary duplicates: 501728  after: 501728
(501728, 19)
   timestamp  category  track_id  translation_x  translation_y  translation_z  \
0    -313.36         1         1        39.0555         7.8648       508.7587   
1    -313.28         1         1        39.0382         7.8646       508.7587   
2    -313.20         1         1        39.0212         7.8643       508.7587   
3    -313.12         1         1        38.9920         7.8640       508.7588   
4    -313.04         1         1        38.9672         7.8641       508.7588   

   dimension_x  dimension_y  dimension_z  rotation_x  rotation_y  rotation_z  \
0       4.8229       2.1169        1.523     -0.0024     -0.0017     -3.0568   
1       4.8229       2.1169        1.523     -0.0024     -0.0017     -3.0570   
2       4.8229       2.1169        1.523     -0.0024     -0.0017     -3.0556   
3       4.8229       2.1169        1.523     -0.0024     -0.0017     -3.0538   
4       4.8229       2.1169        1

In [28]:
# This confirm the car category code from the metadata file.
import os
if os.path.exists(META_PATH):
    with open(META_PATH) as f:
        meta = json.load(f)
    print(meta)

else:
    print(f"No {META_PATH} found, keeping CAR_CATEGORY = {CAR_CATEGORY} (the largest class in the file).")

cars = df[df["category"] == CAR_CATEGORY].copy()
cars = cars.sort_values(["track_id", "timestamp"]).reset_index(drop=True)
print("Rows for cars:", len(cars), "  Unique car tracks:", cars["track_id"].nunique())


{'categoryies': {'car': 1, 'pedestrian': 2, 'bicycle': 3, 'trailer': 4, 'motorcycle': 5, 'truck': 6, 'bus': 7, 'scooter': 8, 'streetcar': 9}, 'coordinates': {'global': 'UTM32N', 'offset_x': 692009.0, 'offset_y': 5338095.0}, 'reference_time': {'0': '14:59:25'}}
Rows for cars: 214997   Unique car tracks: 309


## 2. Turn local drone coordinates into real world coordinates

Each position is <i><b>(UTM easting, UTM northing) = (REF_EASTING + translation_x, REF_NORTHING + translation_y)</i></b>.
This is then converted to latitude and longitude with <i>pyproj</i>, so the trajectories can be lined
up with the OpenStreetMap road network.


In [29]:
cars["utm_x"] = UTM_REF_EASTING + cars["translation_x"]
cars["utm_y"] = UTM_REF_NORTHING + cars["translation_y"]

to_wgs84 = Transformer.from_crs(f"EPSG:326{UTM_ZONE}", "EPSG:4326", always_xy=True)
lon, lat = to_wgs84.transform(cars["utm_x"].values, cars["utm_y"].values)
cars["lon"] = lon
cars["lat"] = lat

# heading in degrees, 0 = north, clockwise, converted from the dataset's rotation_z (radians)
cars["heading_deg"] = (90 - np.degrees(cars["rotation_z"])) % 360

# speed in km/h from the velocity components, useful later as a feature
cars["speed_kmh"] = np.hypot(cars["velocity_x"], cars["velocity_y"]) * 3.6

study_centre_lat = cars["lat"].mean()
study_centre_lon = cars["lon"].mean()
print("Study area centre:", study_centre_lat, study_centre_lon)
print(cars[["timestamp", "track_id", "lat", "lon", "heading_deg", "speed_kmh"]].head())


Study area centre: 48.16718049515299 11.583677045982508
   timestamp  track_id        lat        lon  heading_deg  speed_kmh
0    -313.36         1  48.167131  11.582975   265.141739   0.597210
1    -313.28         1  48.167131  11.582975   265.153198   0.573742
2    -313.20         1  48.167131  11.582975   265.072984   0.548870
3    -313.12         1  48.167131  11.582974   264.969851   0.518946
4    -313.04         1  48.167131  11.582974   264.895367   0.485406


## 3. Munich road network with semantic attributes

Pulls the drive network from OpenStreetMap around the study area, using OSMnx, then reads
off the semantic attributes used : road type, speed limit, number
of lanes, one way flag, road length, bridge, tunnel and junction.

In [30]:
STUDY_RADIUS_M = 700   # TUMDOT-MUC covers about 700 m along Rheinstrasse

def semantic_edges_from_graph(G):
    """Turn an OSMnx graph into a flat table of road segments with semantic attributes."""
    rows = []
    for u, v, key, data in G.edges(keys=True, data=True):
        highway = data.get("highway", "unclassified")
        if isinstance(highway, list):
            highway = highway[0]
        maxspeed = data.get("maxspeed", 50)
        if isinstance(maxspeed, list):
            maxspeed = maxspeed[0]
        try:
            maxspeed = float(str(maxspeed).split()[0])
        except (ValueError, TypeError):
            maxspeed = 50.0
        lanes = data.get("lanes", 1)
        if isinstance(lanes, list):
            lanes = lanes[0]
        try:
            lanes = int(lanes)
        except (ValueError, TypeError):
            lanes = 1
        rows.append({
            "road_id": f"{u}_{v}_{key}",
            "u": u, "v": v, "key": key,
            "highway": highway,
            "maxspeed": maxspeed,
            "lanes": lanes,
            "one_way": bool(data.get("oneway", False)),
            "length_m": float(data.get("length", 0.0)),
            "bridge": bool(data.get("bridge", False)),
            "tunnel": bool(data.get("tunnel", False)),
            "junction": bool(data.get("junction", False)),
            "geometry": data.get("geometry", None),
        })
    return pd.DataFrame(rows)


try:
    G_osm = ox.graph_from_point((study_centre_lat, study_centre_lon),
                                 dist=STUDY_RADIUS_M, network_type="drive", simplify=True)
    G_osm = ox.add_edge_speeds(G_osm)
    G_osm = ox.add_edge_travel_times(G_osm)
    road_graph = G_osm
    edges_df = semantic_edges_from_graph(G_osm)
    print("Loaded real Munich network:", len(road_graph.nodes), "nodes,", len(road_graph.edges), "edges")
    USING_SYNTHETIC_NETWORK = False

except Exception as e:
    print("Could not reach OpenStreetMap (", type(e).__name__, "), building a synthetic fallback grid instead.")
    USING_SYNTHETIC_NETWORK = True

    # A small grid of streets covering the same area as the trajectories, with made up but
    # plausible semantic tags, so the rest of the notebook can still be run and checked.
    lat_span = cars["lat"].max() - cars["lat"].min()
    lon_span = cars["lon"].max() - cars["lon"].min()
    lat0, lon0 = cars["lat"].min() - 0.2 * lat_span, cars["lon"].min() - 0.2 * lon_span
    n_rows, n_cols = 6, 10
    G_syn = nx.MultiDiGraph()
    for r in range(n_rows):
        for c in range(n_cols):
            node_id = r * n_cols + c
            G_syn.add_node(node_id,
                            y=lat0 + r * (1.4 * lat_span / n_rows),
                            x=lon0 + c * (1.4 * lon_span / n_cols))

    highway_types = ["residential", "tertiary", "secondary", "primary"]
    rng = np.random.default_rng(SEED)
    for r in range(n_rows):
        for c in range(n_cols):
            u = r * n_cols + c
            for dr, dc in [(0, 1), (1, 0)]:
                rr, cc = r + dr, c + dc
                if rr < n_rows and cc < n_cols:
                    v = rr * n_cols + cc
                    htype = highway_types[rng.integers(0, len(highway_types))]
                    speed = {"residential": 30, "tertiary": 50, "secondary": 50, "primary": 60}[htype]
                    length = float(rng.uniform(60, 160))
                    for a, b in [(u, v), (v, u)]:
                        G_syn.add_edge(a, b, key=0, highway=htype, maxspeed=speed,
                                       lanes=int(rng.integers(1, 3)), oneway=False,
                                       length=length, bridge=False, tunnel=False,
                                       junction=bool(rng.random() < 0.05))
    road_graph = G_syn
    edges_df = semantic_edges_from_graph(G_syn)
    print("Synthetic network:", len(road_graph.nodes), "nodes,", len(road_graph.edges), "edges")

print(edges_df.head())
print("Road types present:", edges_df["highway"].unique())


Loaded real Munich network: 176 nodes, 431 edges
               road_id        u          v  key      highway  maxspeed  lanes  \
0   1955571_28092168_0  1955571   28092168    0  residential      30.0      2   
1   1955571_27744793_0  1955571   27744793    0  residential      30.0      2   
2  1955571_259667717_0  1955571  259667717    0  residential      30.0      1   
3   1955573_13883759_0  1955573   13883759    0  residential      30.0      1   
4   1955573_54794644_0  1955573   54794644    0  residential      30.0      1   

   one_way    length_m  bridge  tunnel  junction  \
0    False  101.860865   False   False     False   
1    False   91.290226   False   False     False   
2    False   56.189548   False   False     False   
3    False   88.837047   False   False     False   
4    False  120.630866   False   False     False   

                                            geometry  
0  LINESTRING (11.5808563 48.1639796, 11.580207 4...  
1  LINESTRING (11.5808563 48.1639796, 11.

In [31]:
# each road segment's own compass bearing (0 = north, clockwise), used later to compare
# a candidate's direction against the vehicle's GPS heading
bearings = []
for _, edge_row in edges_df.iterrows():
    y1, x1 = road_graph.nodes[edge_row["u"]]["y"], road_graph.nodes[edge_row["u"]]["x"]
    y2, x2 = road_graph.nodes[edge_row["v"]]["y"], road_graph.nodes[edge_row["v"]]["x"]
    bearings.append(math.degrees(math.atan2(x2 - x1, y2 - y1)) % 360)
edges_df["bearing_deg"] = bearings
print(edges_df[["road_id", "highway", "bearing_deg"]].head())


               road_id      highway  bearing_deg
0   1955571_28092168_0  residential   269.303137
1   1955571_27744793_0  residential    99.594980
2  1955571_259667717_0  residential     8.719610
3   1955573_13883759_0  residential   178.578133
4   1955573_54794644_0  residential   352.236254


## 4. Ground truth: match each drone position to the road it is really on

The drone positions are accurate to a few centimetres, so a simple nearest edge search is a
good enough way to build the ground truth road for each point. This ground truth is what the
noisy GPS experiments are checked against.


In [32]:
def edge_midpoints_and_lookup(G):
    """Builds a KD-tree over edge midpoints, in latitude and longitude, for nearest edge search."""
    mids, road_ids = [], []
    for u, v, key, data in G.edges(keys=True, data=True):
        y1, x1 = G.nodes[u]["y"], G.nodes[u]["x"]
        y2, x2 = G.nodes[v]["y"], G.nodes[v]["x"]
        mids.append(((y1 + y2) / 2.0, (x1 + x2) / 2.0))
        road_ids.append(f"{u}_{v}_{key}")
    tree = cKDTree(np.array(mids))
    return tree, road_ids


edge_tree, edge_road_ids = edge_midpoints_and_lookup(road_graph)

query_points = cars[["lat", "lon"]].values
_, nearest_idx = edge_tree.query(query_points, k=1)
cars["ground_truth_road_id"] = [edge_road_ids[i] for i in nearest_idx]

print(cars[["track_id", "timestamp", "lat", "lon", "ground_truth_road_id"]].head())
print("Unique ground truth roads used:", cars["ground_truth_road_id"].nunique())


   track_id  timestamp        lat        lon ground_truth_road_id
0         1    -313.36  48.167131  11.582975  36717026_36717019_0
1         1    -313.28  48.167131  11.582975  36717026_36717019_0
2         1    -313.20  48.167131  11.582975  36717026_36717019_0
3         1    -313.12  48.167131  11.582974  36717026_36717019_0
4         1    -313.04  48.167131  11.582974  36717026_36717019_0
Unique ground truth roads used: 22


## 5. Trajectory level train, validation and test split

Splitting is done by `track_id`, not by row, so that points from the same trip never end up
in two different sets. This helps prevent data leakages


In [33]:
all_tracks = cars["track_id"].unique()
train_tracks, test_tracks = train_test_split(all_tracks, test_size=0.30, random_state=SEED)
val_tracks, test_tracks = train_test_split(test_tracks, test_size=0.50, random_state=SEED)

print("Tracks: train", len(train_tracks), " val", len(val_tracks), " test", len(test_tracks))

split_map = {}
for t in train_tracks:
    split_map[t] = "train"
for t in val_tracks:
    split_map[t] = "val"
for t in test_tracks:
    split_map[t] = "test"
cars["split"] = cars["track_id"].map(split_map)
print(cars["split"].value_counts())


Tracks: train 216  val 46  test 47
split
train    158709
val       30259
test      26029
Name: count, dtype: int64


## 6. Artificial GPS noise and downsampling

Gaussian position noise is added at several standard deviations, and
the 12.5 Hz drone data is downsampled to several sampling intervals, to build the noisy GPS
conditions the map matching methods are tested under.


In [34]:
NOISE_STD_M = [0, 5, 10, 20, 30, 50]     # metres
SAMPLING_INTERVAL_S = [1, 2, 5, 10]      # seconds

# local metres-per-degree conversion, good enough for a 700 m study area
METRES_PER_DEG_LAT = 111_320.0
METRES_PER_DEG_LON = 111_320.0 * math.cos(math.radians(study_centre_lat))


def make_noisy_condition(data, noise_std, interval_s, rng):
    """Downsamples each track to one point roughly every interval_s seconds, then adds
    Gaussian position noise with standard deviation noise_std metres."""
    kept_rows = []
    for track_id, group in data.groupby("track_id"):
        group = group.sort_values("timestamp")
        last_kept_t = None
        for _, row in group.iterrows():
            if last_kept_t is None or row["timestamp"] - last_kept_t >= interval_s:
                kept_rows.append(row)
                last_kept_t = row["timestamp"]
    sampled = pd.DataFrame(kept_rows).reset_index(drop=True)

    if noise_std > 0:
        d_lat = rng.normal(0, noise_std, len(sampled)) / METRES_PER_DEG_LAT
        d_lon = rng.normal(0, noise_std, len(sampled)) / METRES_PER_DEG_LON
        sampled["noisy_lat"] = sampled["lat"] + d_lat
        sampled["noisy_lon"] = sampled["lon"] + d_lon
    else:
        sampled["noisy_lat"] = sampled["lat"]
        sampled["noisy_lon"] = sampled["lon"]
    sampled["noise_std_m"] = noise_std
    sampled["interval_s"] = interval_s
    return sampled


rng = np.random.default_rng(SEED)
noisy_conditions = {}
for noise_std in NOISE_STD_M:
    for interval_s in SAMPLING_INTERVAL_S:
        noisy_conditions[(noise_std, interval_s)] = make_noisy_condition(cars, noise_std, interval_s, rng)

print("Built", len(noisy_conditions), "noise x sampling conditions.")
example = noisy_conditions[(20, 2)]
print("Example condition (20 m noise, 2 s sampling):", len(example), "points")
print(example[["track_id", "timestamp", "lat", "lon", "noisy_lat", "noisy_lon"]].head())


Built 24 noise x sampling conditions.
Example condition (20 m noise, 2 s sampling): 8755 points
   track_id  timestamp        lat        lon  noisy_lat  noisy_lon
0         1    -313.36  48.167131  11.582975  48.167103  11.583171
1         1    -311.36  48.167131  11.582973  48.166860  11.582962
2         1    -309.36  48.167131  11.582973  48.167191  11.582934
3         1    -307.36  48.167131  11.582973  48.166886  11.582739
4         1    -305.36  48.167131  11.582973  48.167428  11.582350


## 7. Baseline 1: nearest road

For every noisy GPS point, this just picks the closest road edge. It is the simplest possible
map matching method and the floor that the other methods are compared against.


In [35]:
def nearest_road_baseline(noisy_df, tree, road_ids):
    pts = noisy_df[["noisy_lat", "noisy_lon"]].values
    _, idx = tree.query(pts, k=1)
    return [road_ids[i] for i in idx]


baseline_results = {}
for cond_key, noisy_df in noisy_conditions.items():
    preds = nearest_road_baseline(noisy_df, edge_tree, edge_road_ids)
    baseline_results[("nearest_road", cond_key)] = pd.Series(preds, index=noisy_df.index)

print("Nearest road baseline computed for", len(NOISE_STD_M) * len(SAMPLING_INTERVAL_S), "conditions.")


Nearest road baseline computed for 24 conditions.


## 8. Baseline 2: FMM style, HMM based map matching

This follows the same idea as FMM (Fast Map Matching). For each point, several nearby roads
are kept as candidates (the emission score, based on distance). A simple Viterbi pass then
picks the sequence of roads that best balances closeness to the GPS points against how
connected the roads are to each other (the transition score), instead of matching each point
on its own.


In [36]:
K_CANDIDATES = 5
CANDIDATE_RADIUS_DEG = 60 / METRES_PER_DEG_LAT   # about 60 m search radius

# precompute simple adjacency: which road ids follow which, from the graph structure
road_id_list = edge_road_ids
road_index = {rid: i for i, rid in enumerate(road_id_list)}
adjacency = {rid: set() for rid in road_id_list}
for u, v, key in road_graph.edges(keys=True):
    rid = f"{u}_{v}_{key}"
    for _, v2, key2 in road_graph.out_edges(v, keys=True):
        adjacency[rid].add(f"{v}_{v2}_{key2}")


def hmm_map_match(noisy_df, tree, road_ids):
    """A proper Viterbi map matcher: per point candidates from a radius search (closest
    first), scored by closeness to the GPS point (emission) and by whether they connect to
    the previous step's candidate (transition), then a full backward trace picks the single
    best road at every step, not just the closest candidate at each step on its own."""
    pts = noisy_df[["noisy_lat", "noisy_lon"]].values

    candidates_per_step = []     # list of (cand_roads, emission_scores) per time step
    for t in range(len(pts)):
        idx = tree.query_ball_point(pts[t], r=CANDIDATE_RADIUS_DEG)
        if len(idx) == 0:
            _, single = tree.query(pts[t], k=1)
            idx = [int(single)]
        dists = np.linalg.norm(np.array([tree.data[i] for i in idx]) - pts[t], axis=1)
        order = np.argsort(dists)                      # closest candidate first
        idx = [idx[i] for i in order][:K_CANDIDATES]
        dists = dists[order][:K_CANDIDATES]
        cand_roads = [road_ids[i] for i in idx]
        candidates_per_step.append((cand_roads, -dists))   # closer -> higher (less negative) score

    # forward pass: build a score table and a backpointer table
    score_table, backpointer_table = [], []
    prev_roads, prev_scores = None, None
    for t, (cand_roads, emission) in enumerate(candidates_per_step):
        if prev_scores is None:
            scores = emission
            backpointers = [None] * len(cand_roads)
        else:
            scores = np.full(len(cand_roads), -1e9)
            backpointers = [0] * len(cand_roads)
            for j, rid in enumerate(cand_roads):
                best_score, best_prev = -1e9, 0
                for pi, prid in enumerate(prev_roads):
                    trans = 0.0 if (rid == prid or rid in adjacency.get(prid, set())) else -5.0
                    candidate_score = prev_scores[pi] + trans + emission[j]
                    if candidate_score > best_score:
                        best_score, best_prev = candidate_score, pi
                scores[j], backpointers[j] = best_score, best_prev
        score_table.append(cand_roads)
        backpointer_table.append(backpointers)
        prev_roads, prev_scores = cand_roads, scores

    # backward trace: start at the best scoring candidate of the last step
    chosen_index = int(np.argmax(prev_scores))
    chosen_roads = [None] * len(candidates_per_step)
    for t in range(len(candidates_per_step) - 1, -1, -1):
        chosen_roads[t] = score_table[t][chosen_index]
        chosen_index = backpointer_table[t][chosen_index] if backpointer_table[t][chosen_index] is not None else 0
    return chosen_roads


hmm_results = {}
for cond_key, noisy_df in noisy_conditions.items():
    preds = hmm_map_match(noisy_df, edge_tree, edge_road_ids)
    hmm_results[("fmm_hmm", cond_key)] = pd.Series(preds, index=noisy_df.index)

print("FMM/HMM baseline computed for all conditions.")


FMM/HMM baseline computed for all conditions.


## 9. Candidate table, with geometric and semantic columns kept separate

For every noisy GPS point, this keeps the nearby road candidates and works out two kinds of
feature: geometric features (distance to the road, the difference between the vehicle heading
and the road's own direction, and whether the candidate connects to the road picked for the
previous point) and semantic features (the ten road attributes from Chapter 4, as far as they
are available from OpenStreetMap). The two kinds of feature are kept in separate columns on
purpose, since the temporal model in the next section will use only the geometric ones, and
the semantic model will use both, with nothing else about the two models changed.


In [37]:
GEOMETRIC_FEATURES = ["distance_m", "heading_diff_deg", "connects_to_prev"]
SEMANTIC_FEATURES = ["highway_id", "maxspeed", "lanes", "one_way", "length_m", "bridge", "tunnel", "junction"]
ALL_FEATURES = GEOMETRIC_FEATURES + SEMANTIC_FEATURES

ROAD_TYPE_VOCAB = sorted(edges_df["highway"].unique().tolist())
road_type_to_int = {t: i for i, t in enumerate(ROAD_TYPE_VOCAB)}
edges_df["highway_id"] = edges_df["highway"].map(road_type_to_int)
edges_lookup = edges_df.set_index("road_id")


def build_candidate_table(noisy_df, tree, road_ids, edges_lookup, k=K_CANDIDATES):
    noisy_df = noisy_df.sort_values(["track_id", "timestamp"]).reset_index(drop=True)
    pts = noisy_df[["noisy_lat", "noisy_lon"]].values
    dists_deg, idxs = tree.query(pts, k=k)

    rows = []
    previous_nearest_road = {}   # one entry per track_id, the closest candidate at the last point
    for t in range(len(noisy_df)):
        row = noisy_df.iloc[t]
        track_id = row["track_id"]
        prev_road = previous_nearest_road.get(track_id)
        cand_idx = np.atleast_1d(idxs[t])
        cand_dist_deg = np.atleast_1d(dists_deg[t])

        for rank, (ci, dd) in enumerate(zip(cand_idx, cand_dist_deg)):
            rid = road_ids[ci]
            info = edges_lookup.loc[rid]
            dist_m = dd * METRES_PER_DEG_LAT
            heading_diff = abs(row["heading_deg"] - info["bearing_deg"]) % 360
            heading_diff = min(heading_diff, 360 - heading_diff)
            connects = 1.0 if prev_road is None or rid == prev_road or rid in adjacency.get(prev_road, set()) else 0.0
            rows.append({
                "point_idx": t,
                "track_id": track_id,
                "split": row["split"],
                "noise_std_m": row["noise_std_m"],
                "interval_s": row["interval_s"],
                "road_id": rid,
                "distance_m": dist_m,
                "heading_diff_deg": heading_diff,
                "connects_to_prev": connects,
                "highway_id": info["highway_id"],
                "maxspeed": info["maxspeed"],
                "lanes": info["lanes"],
                "one_way": float(info["one_way"]),
                "length_m": info["length_m"],
                "bridge": float(info["bridge"]),
                "tunnel": float(info["tunnel"]),
                "junction": float(info["junction"]),
                "is_correct": rid == row["ground_truth_road_id"],
            })
            if rank == 0:
                previous_nearest_road[track_id] = rid
    return pd.DataFrame(rows)


# building the full candidate table for every condition is expensive, so the neural models
# are trained on one representative noise/sampling condition (20 m noise, 2 s sampling) plus
# evaluated on all of them, exactly as chapter 4 describes: models are trained once and then
# tested across the full grid of noise and sampling conditions.
TRAIN_CONDITION = (20, 2)
train_source = noisy_conditions[TRAIN_CONDITION]
candidate_table = build_candidate_table(train_source, edge_tree, edge_road_ids, edges_lookup)
print(candidate_table.shape)
print(candidate_table.head(10))
print("Share of candidates that are correct:", candidate_table["is_correct"].mean())


(43775, 18)
   point_idx  track_id  split  noise_std_m  interval_s                road_id  \
0          0         1  train           20           2    36717019_36717026_0   
1          0         1  train           20           2    36717026_36717019_0   
2          0         1  train           20           2    36717019_36717018_0   
3          0         1  train           20           2    36717018_36717019_0   
4          0         1  train           20           2  1693994297_13883760_0   
5          1         1  train           20           2    36717019_36717026_0   
6          1         1  train           20           2    36717026_36717019_0   
7          1         1  train           20           2    36717019_36717018_0   
8          1         1  train           20           2    36717018_36717019_0   
9          1         1  train           20           2  1693994297_13883760_0   

   distance_m  heading_diff_deg  connects_to_prev  highway_id  maxspeed  \
0   33.291012        

## 10. TensorFlow map matching models: temporal and semantic

Both models below are built by the same function, `build_lstm_model`. The GPS trajectory
always goes through an LSTM layer, exactly as described in Chapter 2 and Chapter 4. The only
difference between the two models is the width of the candidate feature vector passed in
alongside it: the temporal model gets the three geometric columns, and the semantic model
gets the geometric columns plus the eight semantic columns. Nothing else about the model
changes, so any difference in accuracy between the two can only come from the added
semantic attributes.


In [38]:
WINDOW = 5          # number of recent points used as context


def make_windowed_dataset(cand_table, feature_cols, window=WINDOW):
    """Groups candidates by point, and for each candidate builds a short window of the
    same track's preceding candidate rows as sequence context."""
    X_seq, X_cand, y = [], [], []
    grouped = cand_table.groupby(["track_id", "point_idx"])
    for (track_id, point_idx), group in grouped:
        track_hist = cand_table[(cand_table["track_id"] == track_id) &
                                 (cand_table["point_idx"] <= point_idx) &
                                 (cand_table["point_idx"] > point_idx - window)]
        hist_features = track_hist.groupby("point_idx")[GEOMETRIC_FEATURES].mean()
        hist_features = hist_features.reindex(range(point_idx - window + 1, point_idx + 1), fill_value=0.0)
        seq = hist_features[GEOMETRIC_FEATURES].values.astype("float32")
        for _, cand_row in group.iterrows():
            X_seq.append(seq)
            X_cand.append(cand_row[feature_cols].values.astype("float32"))
            y.append(float(cand_row["is_correct"]))
    return np.array(X_seq), np.array(X_cand), np.array(y)


train_rows = candidate_table[candidate_table["split"] == "train"]
val_rows = candidate_table[candidate_table["split"] == "val"]
test_rows = candidate_table[candidate_table["split"] == "test"]

# the temporal model's inputs: candidate_input carries only the three geometric columns
X_seq_train, X_cand_geo_train, y_train = make_windowed_dataset(train_rows, GEOMETRIC_FEATURES)
X_seq_val, X_cand_geo_val, y_val = make_windowed_dataset(val_rows, GEOMETRIC_FEATURES)

# the semantic model's inputs: candidate_input carries the geometric columns plus the semantic ones
_, X_cand_sem_train, _ = make_windowed_dataset(train_rows, ALL_FEATURES)
_, X_cand_sem_val, _ = make_windowed_dataset(val_rows, ALL_FEATURES)

print("Temporal candidate width:", X_cand_geo_train.shape[-1], " Semantic candidate width:", X_cand_sem_train.shape[-1])


Temporal candidate width: 3  Semantic candidate width: 11


In [39]:
def build_lstm_model(gps_steps, gps_features, num_candidate_features):
    gps_input = keras.Input(shape=(gps_steps, gps_features), name="trajectory_window")
    candidate_input = keras.Input(shape=(num_candidate_features,), name="candidate_features")

    lstm_output = layers.LSTM(32)(gps_input)
    combined = layers.Concatenate()([lstm_output, candidate_input])
    x = layers.Dense(32, activation="relu")(combined)
    score = layers.Dense(1, activation="sigmoid", name="correct_road_score")(x)

    model = keras.Model(inputs=[gps_input, candidate_input], outputs=score)
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model


temporal_model = build_lstm_model(gps_steps=WINDOW, gps_features=len(GEOMETRIC_FEATURES),
                                   num_candidate_features=len(GEOMETRIC_FEATURES))
temporal_model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ trajectory_window   │ (None, 5, 3)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 32)        │      4,608 │ trajectory_windo… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ candidate_features  │ (None, 3)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 35)        │          0 │ lstm[0][0],       │
│ (Concatenate)       │                   │            │ candidate_featur… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32)        │      1,152 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ correct_road_score  │ (None, 1)         │         33 │ dense[0][0]       │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,793 (22.63 KB)

 Trainable params: 5,793 (22.63 KB)

 Non-trainable params: 0 (0.00 B)

In [40]:
history_temporal = temporal_model.fit(
    [X_seq_train, X_cand_geo_train], y_train,
    validation_data=([X_seq_val, X_cand_geo_val], y_val),
    epochs=10, batch_size=64, verbose=1,
)


Epoch 1/10
505/505 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - accuracy: 0.8059 - loss: 0.4787 - val_accuracy: 0.8120 - val_loss: 0.4089
Epoch 2/10
505/505 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.8197 - loss: 0.4010 - val_accuracy: 0.8214 - val_loss: 0.3933
Epoch 3/10
505/505 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.8240 - loss: 0.3919 - val_accuracy: 0.8219 - val_loss: 0.3872
Epoch 4/10
505/505 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.8274 - loss: 0.3874 - val_accuracy: 0.8253 - val_loss: 0.3847
Epoch 5/10
505/505 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.8286 - loss: 0.3846 - val_accuracy: 0.8284 - val_loss: 0.3828
Epoch 6/10
505/505 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.8295 - loss: 0.3826 - val_accuracy: 0.8299 - val_loss: 0.3807
Epoch 7/10
505/505 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.8310 - loss: 0.3809 - val_accuracy: 0.8300 - val_loss: 0.3794
Epoch 8/10
505/505 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.8318 - loss: 0.3796 - val_accuracy: 

In [41]:
semantic_model = build_lstm_model(gps_steps=WINDOW, gps_features=len(GEOMETRIC_FEATURES),
                                   num_candidate_features=len(ALL_FEATURES))
semantic_model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ trajectory_window   │ (None, 5, 3)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 32)        │      4,608 │ trajectory_windo… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ candidate_features  │ (None, 11)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 43)        │          0 │ lstm_1[0][0],     │
│ (Concatenate)       │                   │            │ candidate_featur… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 32)        │      1,408 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ correct_road_score  │ (None, 1)         │         33 │ dense_1[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 6,049 (23.63 KB)

 Trainable params: 6,049 (23.63 KB)

 Non-trainable params: 0 (0.00 B)

In [42]:
history_semantic = semantic_model.fit(
    [X_seq_train, X_cand_sem_train], y_train,
    validation_data=([X_seq_val, X_cand_sem_val], y_val),
    epochs=10, batch_size=64, verbose=1,
)


Epoch 1/10
505/505 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.7958 - loss: 0.6096 - val_accuracy: 0.8253 - val_loss: 0.3866
Epoch 2/10
505/505 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.8276 - loss: 0.3848 - val_accuracy: 0.8352 - val_loss: 0.3702
Epoch 3/10
505/505 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.8333 - loss: 0.3729 - val_accuracy: 0.8381 - val_loss: 0.3639
Epoch 4/10
505/505 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.8361 - loss: 0.3660 - val_accuracy: 0.8407 - val_loss: 0.3609
Epoch 5/10
505/505 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.8387 - loss: 0.3609 - val_accuracy: 0.8437 - val_loss: 0.3567
Epoch 6/10
505/505 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.8412 - loss: 0.3565 - val_accuracy: 0.8455 - val_loss: 0.3529
Epoch 7/10
505/505 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.8421 - loss: 0.3534 - val_accuracy: 0.8468 - val_loss: 0.3490
Epoch 8/10
505/505 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.8431 - loss: 0.3505 - val_accuracy: 0.

## 11. Feature ablation study

Section 4.8 describes checking how much each group of semantic attributes actually helps, by
removing one group at a time (zeroing it out) and retraining the semantic model, then reading
off the drop in validation accuracy. `road_type_group` covers highway type and lanes,
`speed_group` covers the speed limit, and `flag_group` covers one way, bridge, tunnel and
junction.


In [43]:
ABLATION_GROUPS = {
    "road_type_group": ["highway_id", "lanes"],
    "speed_group": ["maxspeed"],
    "flag_group": ["one_way", "bridge", "tunnel", "junction"],
}
feature_position = {name: i for i, name in enumerate(ALL_FEATURES)}


def zero_out_columns(array, column_names):
    array = array.copy()
    for name in column_names:
        array[..., feature_position[name]] = 0.0
    return array


ablation_results = {}
full_val_acc = semantic_model.evaluate([X_seq_val, X_cand_sem_val], y_val, verbose=0)[1]
ablation_results["full_model"] = full_val_acc
print("Full model validation accuracy:", full_val_acc)

for group_name, cols in ABLATION_GROUPS.items():
    X_cand_ablated_train = zero_out_columns(X_cand_sem_train, cols)
    X_cand_ablated_val = zero_out_columns(X_cand_sem_val, cols)

    ablated_model = keras.models.clone_model(semantic_model)
    ablated_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    ablated_model.fit(
        [X_seq_train, X_cand_ablated_train], y_train,
        validation_data=([X_seq_val, X_cand_ablated_val], y_val),
        epochs=10, batch_size=64, verbose=0,
    )
    acc = ablated_model.evaluate([X_seq_val, X_cand_ablated_val], y_val, verbose=0)[1]
    ablation_results[group_name] = acc
    print(group_name, "removed -> validation accuracy", acc, " drop of", full_val_acc - acc)

ablation_table = pd.DataFrame(
    [{"group_removed": k, "val_accuracy": v, "drop_from_full": full_val_acc - v}
     for k, v in ablation_results.items() if k != "full_model"]
)
print(ablation_table)


Full model validation accuracy: 0.8487012982368469
road_type_group removed -> validation accuracy 0.84188312292099  drop of 0.006818175315856934
speed_group removed -> validation accuracy 0.8357142806053162  drop of 0.012987017631530762
flag_group removed -> validation accuracy 0.8271104097366333  drop of 0.021590888500213623
     group_removed  val_accuracy  drop_from_full
0  road_type_group      0.841883        0.006818
1      speed_group      0.835714        0.012987
2       flag_group      0.827110        0.021591


## 12. Next road segment prediction

Section 3.7 and Chapter 4 also cover predicting the next road a vehicle will take, given the
roads it has already been matched to. Two models are built: a simple Markov baseline (looks
only at the current road), and a simple recurrent neural network that reads a short history of
road ids and also uses the current road's semantic attributes.


In [44]:
# build a sequence of matched roads per track, from the ground truth road ids computed earlier
road_sequences = (
    cars.sort_values(["track_id", "timestamp"])
    .groupby("track_id")["ground_truth_road_id"]
    .apply(lambda s: [r for i, r in enumerate(s) if i == 0 or r != s.iloc[i - 1]])  # collapse repeats
    .to_dict()
)
sequence_lengths = [len(v) for v in road_sequences.values()]
print("Number of tracks:", len(road_sequences), " average sequence length:", np.mean(sequence_lengths))


Number of tracks: 309  average sequence length: 4.8705501618122975


In [45]:
# Markov baseline: for every road, count what road came right after it in the training tracks
transition_counts = {}
train_id_set = set(train_tracks)
for track_id, seq in road_sequences.items():
    if track_id not in train_id_set:
        continue
    for a, b in zip(seq[:-1], seq[1:]):
        transition_counts.setdefault(a, {}).setdefault(b, 0)
        transition_counts[a][b] += 1


def markov_predict_topk(current_road, k=3):
    options = transition_counts.get(current_road, {})
    ranked = sorted(options.items(), key=lambda kv: kv[1], reverse=True)
    return [r for r, _ in ranked[:k]]


# evaluate on test tracks: for every step, is the true next road in the top 1 / top 3 / top 5
def evaluate_next_segment(predict_fn, track_ids, max_k=5):
    hits = {1: 0, 3: 0, 5: 0}
    reciprocal_ranks = []
    total = 0
    for track_id in track_ids:
        seq = road_sequences.get(track_id, [])
        for a, b in zip(seq[:-1], seq[1:]):
            ranked = predict_fn(a, k=max_k)
            total += 1
            if b in ranked:
                rank = ranked.index(b) + 1
                reciprocal_ranks.append(1.0 / rank)
                for k in hits:
                    if rank <= k:
                        hits[k] += 1
            else:
                reciprocal_ranks.append(0.0)
    result = {f"top_{k}_accuracy": hits[k] / total if total else 0.0 for k in hits}
    result["mrr"] = float(np.mean(reciprocal_ranks)) if reciprocal_ranks else 0.0
    result["n_steps"] = total
    return result


markov_metrics = evaluate_next_segment(markov_predict_topk, test_tracks)
print("Markov baseline:", markov_metrics)


Markov baseline: {'top_1_accuracy': 0.654320987654321, 'top_3_accuracy': 0.9567901234567902, 'top_5_accuracy': 0.9753086419753086, 'mrr': 0.8040123456790124, 'n_steps': 162}


In [46]:
# simple RNN model: embeds road ids, adds the current road's semantic attributes, and
# predicts a probability for every road id that has appeared in the training sequences
NEXT_SEQ_LEN = 4
all_roads_in_sequences = sorted({r for seq in road_sequences.values() for r in seq})
road_to_int = {r: i + 1 for i, r in enumerate(all_roads_in_sequences)}   # 0 is the padding id
vocab_size = len(road_to_int) + 1


def build_next_segment_dataset(track_ids):
    X_ids, X_feat, y_ids = [], [], []
    for track_id in track_ids:
        seq = road_sequences.get(track_id, [])
        int_seq = [road_to_int[r] for r in seq if r in road_to_int]
        for i in range(1, len(int_seq)):
            window_ids = int_seq[max(0, i - NEXT_SEQ_LEN):i]
            window_ids = [0] * (NEXT_SEQ_LEN - len(window_ids)) + window_ids
            current_road = seq[i - 1]
            info = edges_lookup.loc[current_road] if current_road in edges_lookup.index else None
            feat = [0.0, 0.0, 0.0, 0.0] if info is None else \
                [info["highway_id"], info["maxspeed"], info["lanes"], float(info["one_way"])]
            X_ids.append(window_ids)
            X_feat.append(feat)
            y_ids.append(int_seq[i])
    return np.array(X_ids), np.array(X_feat, dtype="float32"), np.array(y_ids)


Xid_train, Xfeat_train, y_next_train = build_next_segment_dataset(train_tracks)
Xid_val, Xfeat_val, y_next_val = build_next_segment_dataset(val_tracks)
Xid_test, Xfeat_test, y_next_test = build_next_segment_dataset(test_tracks)
print(Xid_train.shape, Xfeat_train.shape, y_next_train.shape)


(828, 4) (828, 4) (828,)


In [47]:
id_input = keras.Input(shape=(NEXT_SEQ_LEN,), name="road_id_sequence")
feat_input = keras.Input(shape=(4,), name="current_road_features")

road_embedding = layers.Embedding(vocab_size, 16, mask_zero=True)(id_input)
rnn_output = layers.SimpleRNN(32)(road_embedding)
combined_next = layers.Concatenate()([rnn_output, feat_input])
x_next = layers.Dense(32, activation="relu")(combined_next)
next_road_logits = layers.Dense(vocab_size, activation="softmax", name="next_road")(x_next)

next_segment_model = keras.Model(inputs=[id_input, feat_input], outputs=next_road_logits,
                                  name="rnn_next_segment_model")
next_segment_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
next_segment_model.summary()


Model: "rnn_next_segment_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ road_id_sequence    │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 4, 16)     │        368 │ road_id_sequence… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 4)         │          0 │ road_id_sequence… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn          │ (None, 32)        │      1,568 │ embedding[0][0],  │
│ (SimpleRNN)         │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ current_road_featu… │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, 36)        │          0 │ simple_rnn[0][0], │
│ (Concatenate)       │                   │            │ current_road_fea… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 32)        │      1,184 │ concatenate_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ next_road (Dense)   │ (None, 23)        │        759 │ dense_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,879 (15.15 KB)

 Trainable params: 3,879 (15.15 KB)

 Non-trainable params: 0 (0.00 B)

In [48]:
history_next = next_segment_model.fit(
    [Xid_train, Xfeat_train], y_next_train,
    validation_data=([Xid_val, Xfeat_val], y_next_val),
    epochs=15, batch_size=64, verbose=1,
)


Epoch 1/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 6s 250ms/step - accuracy: 0.0036 - loss: 8.4881 - val_accuracy: 0.0000e+00 - val_loss: 6.4460
Epoch 2/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.1594 - loss: 5.5351 - val_accuracy: 0.1893 - val_loss: 4.5331
Epoch 3/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.2126 - loss: 4.2132 - val_accuracy: 0.1602 - val_loss: 3.6778
Epoch 4/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.1365 - loss: 3.5318 - val_accuracy: 0.2233 - val_loss: 3.1661
Epoch 5/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.2705 - loss: 3.0111 - val_accuracy: 0.3641 - val_loss: 2.7040
Epoch 6/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.4203 - loss: 2.5541 - val_accuracy: 0.4175 - val_loss: 2.3224
Epoch 7/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.4432 - loss: 2.1699 - val_accuracy: 0.4466 - val_loss: 1.9637
Epoch 8/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.5133 - loss: 1.8349 - val_accuracy: 0.548

In [49]:
def rnn_predict_topk(current_road, k=5, window_ids=None):
    # a light wrapper so this can be scored with the same evaluate_next_segment function,
    # using single-step windows of just the current road (padded)
    ids = [0] * (NEXT_SEQ_LEN - 1) + [road_to_int.get(current_road, 0)]
    info = edges_lookup.loc[current_road] if current_road in edges_lookup.index else None
    feat = [0.0, 0.0, 0.0, 0.0] if info is None else \
        [info["highway_id"], info["maxspeed"], info["lanes"], float(info["one_way"])]
    probs = next_segment_model.predict(
        [np.array([ids]), np.array([feat], dtype="float32")], verbose=0
    )[0]
    top_ids = np.argsort(probs)[::-1][:k]
    int_to_road = {v: r for r, v in road_to_int.items()}
    return [int_to_road[i] for i in top_ids if i in int_to_road]


rnn_metrics = evaluate_next_segment(rnn_predict_topk, test_tracks)
print("RNN next segment model:", rnn_metrics)


RNN next segment model: {'top_1_accuracy': 0.3950617283950617, 'top_3_accuracy': 0.7407407407407407, 'top_5_accuracy': 0.8333333333333334, 'mrr': 0.5522633744855967, 'n_steps': 162}


## 13. Evaluation across all noise and sampling conditions

This fills in the table shape from section 5.1: one row per model, per noise level, per
sampling interval, with accuracy, F1 and mean positional error. The four map matching methods
(nearest road, FMM/HMM, temporal LSTM, semantic LSTM) are each run again on every noise and
sampling condition, not just the one they were trained on, to see how they hold up as the GPS
gets noisier and sparser.


In [50]:
from sklearn.metrics import f1_score


def positional_error_m(pred_road_ids, true_road_ids):
    """Mean straight-line distance, in metres, between the midpoint of the predicted road
    and the midpoint of the true road, 0 when they are the same road."""
    errors = []
    for pred, true in zip(pred_road_ids, true_road_ids):
        if pred == true:
            errors.append(0.0)
            continue
        pu, pv, pk = pred.split("_")
        tu, tv, tk = true.split("_")
        pu, pv, tu, tv = int(pu), int(pv), int(tu), int(tv)
        p_lat = (road_graph.nodes[pu]["y"] + road_graph.nodes[pv]["y"]) / 2
        p_lon = (road_graph.nodes[pu]["x"] + road_graph.nodes[pv]["x"]) / 2
        t_lat = (road_graph.nodes[tu]["y"] + road_graph.nodes[tv]["y"]) / 2
        t_lon = (road_graph.nodes[tu]["x"] + road_graph.nodes[tv]["x"]) / 2
        d = math.hypot((p_lat - t_lat) * METRES_PER_DEG_LAT, (p_lon - t_lon) * METRES_PER_DEG_LON)
        errors.append(d)
    return float(np.mean(errors)) if errors else float("nan")


def score_neural_model(model, noisy_df, feature_cols):
    cand_tbl = build_candidate_table(noisy_df, edge_tree, edge_road_ids, edges_lookup)
    X_seq, X_cand, _ = make_windowed_dataset(cand_tbl, feature_cols)
    scores = model.predict([X_seq, X_cand], verbose=0).flatten()
    cand_tbl = cand_tbl.reset_index(drop=True)
    cand_tbl["score"] = scores
    chosen = cand_tbl.loc[cand_tbl.groupby("point_idx")["score"].idxmax()]
    return chosen.sort_values("point_idx")


results_table = []
for (noise_std, interval_s), noisy_df in noisy_conditions.items():
    true_roads = noisy_df["ground_truth_road_id"].values

    nearest_preds = nearest_road_baseline(noisy_df, edge_tree, edge_road_ids)
    hmm_preds = hmm_map_match(noisy_df, edge_tree, edge_road_ids)
    temporal_chosen = score_neural_model(temporal_model, noisy_df, GEOMETRIC_FEATURES)
    semantic_chosen = score_neural_model(semantic_model, noisy_df, ALL_FEATURES)

    for model_name, preds in [
        ("nearest_road", nearest_preds),
        ("fmm_hmm", hmm_preds),
        ("temporal_lstm", temporal_chosen["road_id"].values),
        ("semantic_lstm", semantic_chosen["road_id"].values),
    ]:
        preds = list(preds)
        compare_true = list(true_roads)[:len(preds)]
        accuracy = float(np.mean([p == t for p, t in zip(preds, compare_true)]))
        f1 = f1_score(compare_true, preds, average="micro") if len(set(compare_true)) > 1 else accuracy
        mean_err = positional_error_m(preds, compare_true)
        results_table.append({
            "model": model_name, "noise_m": noise_std, "sampling_s": interval_s,
            "accuracy": accuracy, "f1": f1, "mean_error_m": mean_err,
        })

results_df = pd.DataFrame(results_table)
print(results_df.head(20))
results_df.to_csv("map_matching_results.csv", index=False)
print("Saved map_matching_results.csv")


            model  noise_m  sampling_s  accuracy        f1  mean_error_m
0    nearest_road        0           1  1.000000  1.000000      0.000000
1         fmm_hmm        0           1  0.847035  0.847035      2.111061
2   temporal_lstm        0           1  0.644181  0.644181      9.379724
3   semantic_lstm        0           1  0.676740  0.676740      9.134360
4    nearest_road        0           2  1.000000  1.000000      0.000000
5         fmm_hmm        0           2  0.797944  0.797944      2.846737
6   temporal_lstm        0           2  0.632324  0.632324      9.858390
7   semantic_lstm        0           2  0.656882  0.656882      9.713172
8    nearest_road        0           5  1.000000  1.000000      0.000000
9         fmm_hmm        0           5  0.701095  0.701095      5.089866
10  temporal_lstm        0           5  0.634016  0.634016     10.765399
11  semantic_lstm        0           5  0.639910  0.639910     10.853527
12   nearest_road        0          10  1.000000  1

In [51]:
# Wilcoxon signed rank test, comparing the semantic model against the temporal model, matched
# by noise and sampling condition, exactly as section 4.8 describes
temporal_acc = results_df[results_df["model"] == "temporal_lstm"].sort_values(["noise_m", "sampling_s"])["accuracy"].values
semantic_acc = results_df[results_df["model"] == "semantic_lstm"].sort_values(["noise_m", "sampling_s"])["accuracy"].values

if len(temporal_acc) == len(semantic_acc) and len(temporal_acc) > 0:
    stat, p_value = wilcoxon(semantic_acc, temporal_acc)
    print("Wilcoxon signed rank test, semantic model vs temporal model, accuracy across conditions:")
    print("statistic =", stat, " p-value =", p_value)
else:
    print("Not enough matched conditions to run the Wilcoxon test.")


Wilcoxon signed rank test, semantic model vs temporal model, accuracy across conditions:
statistic = 0.0  p-value = 1.1920928955078125e-07


In [52]:
next_segment_table = pd.DataFrame([
    {"model": "markov_baseline", **markov_metrics},
    {"model": "rnn_model", **rnn_metrics},
])
next_segment_table.to_csv("next_segment_results.csv", index=False)
print(next_segment_table)


             model  top_1_accuracy  top_3_accuracy  top_5_accuracy       mrr  \
0  markov_baseline        0.654321        0.956790        0.975309  0.804012   
1        rnn_model        0.395062        0.740741        0.833333  0.552263   

   n_steps  
0      162  
1      162  
